In [1]:
!pip install comet_ml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.6/796.6 kB 15.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.7 MB/s eta 0:00:00
  Attempting uninstall: python-box
    Found existing installation: python-box 7.4.1
    Uninstalling python-box-7.4.1:
      Successfully uninstalled python-box-7.4.1


In [3]:
!git clone -b lensless-base https://github.com/DommeUse/Lensless-Computational-Imaging.git
%cd Lensless-Computational-Imaging
!python -m pip install virtualenv
!python -m virtualenv /kaggle/working/lensless_env
!/kaggle/working/lensless_env/bin/pip install -r requirements.txt

Cloning into 'Lensless-Computational-Imaging'...
remote: Enumerating objects: 665, done.
remote: Counting objects: 100% (245/245), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 665 (delta 130), reused 124 (delta 63), pack-reused 420 (from 1)
Receiving objects: 100% (665/665), 336.98 KiB | 11.62 MiB/s, done.
Resolving deltas: 100% (304/304), done.
/kaggle/working/Lensless-Computational-Imaging/Lensless-Computational-Imaging
created virtual environment CPython3.12.13.final.0-64-x86_64 in 255ms
  creator CPython3Posix(dest=/kaggle/working/lensless_env, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: Adafruit_PureIO==1.1.11, accelerate==1.14.0, adafruit_blinka==9.1.0, adafruit_circuitpython_busdevice==5.2.17, adafruit_circuitpython_connectionmanager==3.1.8, adafruit_circuitpython_framebuf==1.6.12, adafruit_circuitpython_pcd8544==1.2.23, adafruit

In [4]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["COMET_API_KEY"] = UserSecretsClient().get_secret("COMET_API_KEY")
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["MPLBACKEND"] = "Agg"

# Unrolled ADMM-20

In [5]:
import comet_ml

api = comet_ml.API(api_key=UserSecretsClient().get_secret("COMET_API_KEY"))
exp = api.get_experiment("german-zverev", "lensless-computational-imaging", "yor68kxc3rq083std9sp4ievxefzd6jn")

assets = exp.get_asset_list()
model_asset = [a for a in assets if "model_best" in a['fileName']]

asset_id = model_asset[0]['assetId']
asset_binary = exp.get_asset(asset_id)

with open("./model_best.pth", "wb") as f:
    f.write(asset_binary)

import torch

checkpoint = torch.load("./model_best.pth", "cuda", weights_only = False)
checkpoint['config']['writer']['run_id'] = 'yor68kxc3rq083std9sp4ievxefzd6jn'
torch.save(checkpoint, './model_best.pth')

In [6]:
checkpoint = torch.load("./model_best.pth", "cuda", weights_only = False)
checkpoint['config']['writer']['run_id']

'yor68kxc3rq083std9sp4ievxefzd6jn'

In [ ]:
!/kaggle/working/lensless_env/bin/python -u train.py \
  --config-name=admm_unrolled \
  trainer.save_dir="/kaggle/working/saved" \
  writer.log_checkpoints=True \
  writer.run_name="Unrolled ADMM train" \
  +writer.run_id="yor68kxc3rq083std9sp4ievxefzd6jn" \
  trainer.resume_from="./model_best.pth"

Logging git commit and patch...
COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET INFO: Experiment is live on comet.com https://www.comet.com/german-zverev/lensless-computational-imaging/yor68kxc3rq083std9sp4ievxefzd6jn

README.md: 100%|██████████████████████████████| 478/478 [00:00<00:00, 1.87MB/s]
data/train-00000-of-00009.parquet: 100%|████| 450M/450M [00:05<00:00, 87.9MB/s]
data/train-00001-of-00009.parquet: 100%|█████| 451M/451M [00:04<00:00, 103MB/s]
data/train-00002-of-00009.parquet: 100%|████| 451M/451M [00:10<00:00, 42.2MB/s]
data/train-00003-of-00009.parquet: 100%|████| 448M/448M [00:04<00:00, 89.7MB/s]
data/train-00004-of-00009.parquet: 100%|████| 451M/451M [00:08<00:00, 53.3MB/s]
data/train-00005-of-00009.parquet: 100%|████| 448M/448M [00:04<00:00, 93.5MB/s]
data/train-00006-of-00009.parquet: 100%|█████| 449M/449M [00:03<00:00, 115MB/s]
data/train-00007-of-00009.parquet: 100%|████| 450M/450M [00:04<00:00, 98.9MB/